In [1]:
import pandas as pd 
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import glob
from sklearn.model_selection import train_test_split
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import classification_report, f1_score, accuracy_score
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, auc, roc_curve
from sklearn.preprocessing import LabelEncoder, StandardScaler, MinMaxScaler, OrdinalEncoder
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier, AdaBoostClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.tree import DecisionTreeClassifier
import matplotlib.pyplot as plt
from statsmodels.stats.outliers_influence import variance_inflation_factor  
import warnings
import pickle
import os
import json


warnings.filterwarnings("ignore")
pd.set_option('display.max_columns', 200)
pd.set_option('display.max_rows', 200)

In [2]:
cols = ['ts', 'uid', 'id.orig_h', 'id.orig_p',
        'id.resp_h', 'id.resp_p', 'proto', 'service',
        'duration',  'orig_bytes', 'resp_bytes',
        'conn_state', 'local_orig', 'local_resp',
        'missed_bytes',  'history', 'orig_pkts',
        'orig_ip_bytes', 'resp_pkts', 'resp_ip_bytes',
        'tunnel_parents', 'label']

In [3]:
out_dir = f'/home/meryem.janati/lustre/nlp_team-um6p-st-sccs-id7fz1zvotk/IDS/janati/IDS/Datasets/unsw/Zeek/17-02-2015/timeout60'
df1 = pd.read_csv(out_dir+"/UNSW-NB15_zeek_60.csv")

In [4]:
df1.columns

Index(['ts', 'uid', 'id.orig_h', 'id.orig_p', 'id.resp_h', 'id.resp_p',
       'proto', 'service', 'duration', 'orig_bytes', 'resp_bytes',
       'conn_state', 'local_orig', 'local_resp', 'missed_bytes', 'history',
       'orig_pkts', 'orig_ip_bytes', 'resp_pkts', 'resp_ip_bytes',
       'tunnel_parents', 'id', 'Attack'],
      dtype='object')

In [14]:
out_dir = f'/home/meryem.janati/lustre/nlp_team-um6p-st-sccs-id7fz1zvotk/IDS/janati/IDS/Datasets/unsw/Zeek/22-01-2015/timeout1/UNSW-NB15_zeek_1_v2.csv'

df2 = pd.read_csv(out_dir)
df2.shape

(2058155, 22)

In [15]:
df2

,ts,uid,id.orig_h,id.orig_p,id.resp_h,id.resp_p,proto,service,duration,orig_bytes,resp_bytes,conn_state,local_orig,local_resp,missed_bytes,history,orig_pkts,orig_ip_bytes,resp_pkts,resp_ip_bytes,tunnel_parents,Attack
0,1.421971e+09,ConhGr2Dljccb2ZH1,149.171.126.2,36831,59.166.0.9,64191,tcp,-,0.011938,5200,0,SF,F,F,0,DTaFfA,12,11024,8,416,-,Benign
1,1.421971e+09,CWcyuB160Tahux4Am6,59.166.0.1,3009,149.171.126.0,5190,tcp,-,0.020171,52,53,SF,F,F,0,DTadtfF,4,312,8,522,-,Benign
2,1.421971e+09,CAxoYH6NJYdYTx5oi,59.166.0.0,15716,149.171.126.5,143,tcp,-,0.003997,0,95,SF,F,F,0,^dtfFa,2,104,6,502,-,Benign
3,1.421971e+09,C2Kadf26ujtsBLaJH1,59.166.0.9,1954,149.171.126.1,28062,tcp,-,0.053220,235,19644,SF,F,F,0,ShADTadttfF,56,3390,60,42416,-,Benign
4,1.421971e+09,CMwZqb1HUmTb568nvi,59.166.0.5,47030,149.171.126.2,25,tcp,-,0.080234,2726,95,SF,F,F,0,DTadtAfF,10,5972,14,918,-,Benign
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2058150,1.424230e+09,CxoHoG46BF1Y9pt5id,175.45.176.0,9171,149.171.126.14,110,tcp,pop3,0.753626,57,1845,SF,F,F,0,ShAdtDTFfa,20,946,22,4586,-,Exploits
2058151,1.424230e+09,CsGvgVaB4uKPgVhpk,175.45.176.2,37048,149.171.126.12,143,tcp,-,0.820972,124,1218,SF,F,F,0,ShAdtDTFfa,18,1000,18,3172,-,Exploits
2058152,1.421928e+09,CB17YU1R6855npUGf3,149.171.126.15,179,175.45.176.3,41558,tcp,-,0.074259,0,0,SF,F,F,0,AfFa,6,240,4,160,-,direction_flip:Fuzzers
2058153,1.421928e+09,Cy7gOs1ZYZOwWS33ge,149.171.126.16,1027,175.45.176.0,42174,tcp,dce_rpc,0.244165,60,6848,SF,F,F,0,DTdtFfaA,6,360,14,14256,-,direction_flip:Exploits


In [32]:
df = pd.concat([df1, df2], ignore_index=True, sort=False)

In [33]:
df.shape

(2056606, 23)

In [34]:
df

,ts,uid,id.orig_h,id.orig_p,id.resp_h,id.resp_p,proto,service,duration,orig_bytes,resp_bytes,conn_state,local_orig,local_resp,missed_bytes,history,orig_pkts,orig_ip_bytes,resp_pkts,resp_ip_bytes,tunnel_parents,id,Attack
0,1.424223e+09,C9dAU041N3HqL5K6kd,59.166.0.7,61953,149.171.126.2,24075,tcp,-,0.05984,236,0,S0,F,F,0,Sa,2,112,2,104,-,0,Benign
1,1.424223e+09,CJ1IZu1BweGN34fIia,149.171.126.5,6881,59.166.0.4,65360,tcp,-,0.122888,64154,68,SF,F,F,0,DTaTdtAFf,68,81674,37,2009,-,1,Benign
2,1.424223e+09,CCur3k3zLi0KpW626,59.166.0.9,24935,149.171.126.2,5190,tcp,-,0.004947,216,814,SF,F,F,0,ShADTdtfFa,12,1064,12,2260,-,2,Benign
3,1.424223e+09,CWTE0433ZFQyGfmTzl,59.166.0.6,37415,149.171.126.1,143,tcp,-,0.137649,68,225,SF,F,F,0,DTadtAfF,8,502,14,1094,-,3,Benign
4,1.424223e+09,CZylbB1X08yVSPfALe,59.166.0.7,25718,149.171.126.2,23652,tcp,-,0.074317,235,22843,SF,F,F,3967,ShdGtAgtfFa,32,1672,35,40046,-,4,Benign
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2056601,1.421932e+09,CnYWe54BO49XHsW88h,175.45.176.0,1034,149.171.126.11,520,udp,-,0.000012,1048,0,S0,F,F,0,D,2,1104,0,0,-,1028277,Fuzzers
2056602,1.421928e+09,C9ZdaR3kpQ1BmrbpU7,149.171.126.15,179,175.45.176.3,41558,tcp,-,0.074259,0,0,SF,F,F,0,AfFa,6,240,4,160,-,227010,direction_flip:Fuzzers
2056603,1.421928e+09,COGrCkrzqYLs9qGj7,149.171.126.16,1027,175.45.176.0,42174,tcp,dce_rpc,0.244165,60,6848,SF,F,F,0,DTdtFfaA,6,360,14,14256,-,227012,direction_flip:Exploits
2056604,1.421931e+09,C4kLLs41TF8OZnO1K8,149.171.126.15,89,175.45.176.0,28433,tcp,-,0.021418,0,0,SF,F,F,0,FfaA,4,160,4,160,-,343402,direction_flip:Exploits


In [4]:
def save_scores(timeout, meanScores, stdScore):
    results = {
        'Timeout': timeout,
        'Mean of all scores': meanScores,
        'Std of all Scores': stdScores

    }

    with open(f'../Checkpoints/ET/ET_unsw_zeek_{timeout}.json', 'w') as f:
        json.dump(results, f, indent=4)

In [27]:
timeouts = ['default', 0.5, 1, 2, 3, 4, 5, 6, 10, 30, 60]

In [17]:
timeouts = [1, 5, 60]

# Training

In [18]:
best_f1 = 0
worst_f1 = 1
best_mean, worst_mean, best_std, worst_std = None, None, None, None

save=False

for timeout in timeouts:
    print("Processing timeout : ", timeout)
    out_dir = f'/home/meryem.janati/lustre/nlp_team-um6p-st-sccs-id7fz1zvotk/IDS/janati/IDS/Datasets/unsw/Zeek/22-01-2015/timeout{timeout}/UNSW-NB15_zeek_{timeout}_v2.csv'

    df = pd.read_csv(out_dir)
    
    df = df.drop(columns=['uid', 'id.orig_h', 'id.resp_h', 'tunnel_parents']) # tunnel_parents is empty
    #df = df[~df['Attack'].str.startswith('direction_flip')]


    # Handle missing values (if any)
    df.replace({'orig_bytes': '-'}, '0', inplace=True)
    df['orig_bytes'] = pd.to_numeric(df['orig_bytes'], errors='coerce')
    df['orig_bytes'] = df['orig_bytes'].fillna(0).astype('int64')

    df.replace({'resp_bytes': '-'}, '0', inplace=True)
    df['resp_bytes'] = pd.to_numeric(df['resp_bytes'], errors='coerce')
    df['resp_bytes'] = df['resp_bytes'].fillna(0).astype('int64')


    df.replace({'duration': '-'}, '0', inplace=True)
    df['duration'] = pd.to_numeric(df['duration'], errors='coerce')
    df['duration'] = df['duration'].fillna(0).astype('float64')

    df['service'] = df['service'].replace('-', np.nan)
    df['history'] = df['history'].replace('-', np.nan)
    
    # Convert categorical variables to numerical using Label Encoding
    # Encode protocol and service types
    label_encoders = {}
    for column in ['proto', 'service', 'conn_state', 'history', 'local_orig', 'local_resp']:
        if column in df.columns:
            le = LabelEncoder()
            df[column] = le.fit_transform(df[column])
            label_encoders[column] = le

    # Split df into features and labels
    X = df.drop(columns=['Attack'])  # Assuming 'label' is the target variable
    y = df['Attack']
    
    accuracy, f1, precision, recall =[], [], [], []
    
    skf= StratifiedKFold(n_splits=5,random_state=None)
    skf.get_n_splits(X,y)
    
    for (train_index, test_index), i in zip(skf.split(X, y), range(5)):
        X_train,X_test=X.iloc[train_index],X.iloc[test_index]
        y_train,y_test=y.iloc[train_index],y.iloc[test_index]


        le = LabelEncoder()
        y_train = le.fit_transform(y_train)
        
        
        # Map unseen values to '<unknown>'
        y_test = y_test.map(lambda s: '<unknown>' if s not in le.classes_ else s)

        # Add '<unknown>' to classes and transform
        le.classes_ = np.append(le.classes_, '<unknown>')
        y_test = le.transform(y_test)
        
        
        scaler = StandardScaler()
        X_train = scaler.fit_transform(X_train)
        X_test = scaler.transform(X_test)

        # Initialize and train Extra Trees Classifier
        clf = ExtraTreesClassifier(n_estimators=100, random_state=42) #RandomForestClassifier(random_state=42, class_weight='balanced') 
        clf.fit(X_train, y_train)
        pred = clf.predict(X_test)
        
        f1Score = f1_score(y_true=y_test, y_pred=pred, average='macro')
        accScore=accuracy_score(y_test, pred)
        precScore = precision_score(y_test, pred, average='macro')
        recScrore = recall_score(y_test, pred, average='macro')
                             
        f1.append(f1Score)
        accuracy.append(accScore)
        precision.append(precScore)
        recall.append(recScrore)
        print('Fold: ', i, 'done!')

    meanScores, stdScores = {}, {}
    
    meanScores['f1Mean'] = np.array(f1).mean()
    meanScores['accMean'] = np.array(accuracy).mean()
    meanScores['recMean'] = np.array(recall).mean()
    meanScores['precMean'] = np.array(precision).mean()
    
    stdScores['f1Std'] = np.array(f1).std()
    stdScores['accStd'] = np.array(accuracy).std()
    stdScores['recStd'] = np.array(recall).std()
    stdScores['precStd'] = np.array(precision).std()
    
    print("Mean of all scores: ", meanScores)
    print("Std of all scores: ", stdScores)


    if save:
        save_scores(timeout, meanScores, stdScores)

    if meanScores['f1Mean'] > best_f1: 
        best_timeout = timeout
        best_mean = meanScores
        best_std = stdScores
        best_f1 = meanScores['f1Mean']
    
    if meanScores['f1Mean'] <= worst_f1: 
        
        worst_timeout = timeout
        worst_mean = meanScores
        worst_std = stdScores
        worst_f1 = meanScores['f1Mean']
               
    print('_______________________________________________')

Processing timeout :  1
Fold:  0 done!
Fold:  1 done!
Fold:  2 done!
Fold:  3 done!
Fold:  4 done!
Mean of all scores:  {'f1Mean': 0.43351330571016816, 'accMean': 0.9757841367632661, 'recMean': 0.4195482746490485, 'precMean': 0.5311934552235694}
Std of all scores:  {'f1Std': 0.10980931228774685, 'accStd': 0.006280611925136645, 'recStd': 0.12704602285286065, 'precStd': 0.06885538991845497}
_______________________________________________
Processing timeout :  5
Fold:  0 done!
Fold:  1 done!
Fold:  2 done!
Fold:  3 done!
Fold:  4 done!
Mean of all scores:  {'f1Mean': 0.4350309677867674, 'accMean': 0.9758690376787108, 'recMean': 0.4228613079225426, 'precMean': 0.5203609277156704}
Std of all scores:  {'f1Std': 0.12413670005826499, 'accStd': 0.0063720632886109674, 'recStd': 0.13578787855236843, 'precStd': 0.08537424429166231}
_______________________________________________
Processing timeout :  60
Fold:  0 done!
Fold:  1 done!
Fold:  2 done!
Fold:  3 done!
Fold:  4 done!
Mean of all scores: 

In [15]:
def load_score(timeout):
    with open(f'../Checkpoints/ET/ET_unsw_zeek_{timeout}.json', 'r') as f:
        loaded_results = json.load(f)
    return loaded_results


In [24]:
timeouts = ['default', 0.5]

best_f1 = 0
worst_f1 = 1



for timeout in timeouts:
    loaded_results = load_score(timeout)
    print(loaded_results)
        
        
    if loaded_results['Mean of all scores']['f1Mean'] > best_f1: 
        best_timeout = loaded_results['Timeout']
        best_mean = loaded_results['Mean of all scores']
        best_std = loaded_results['Std of all Scores']
        best_f1 = loaded_results['Mean of all scores']['f1Mean'] 
    
    if loaded_results['Mean of all scores']['f1Mean'] <= worst_f1: 
        
        worst_timeout = loaded_results['Timeout']
        worst_mean = loaded_results['Mean of all scores']
        worst_std = loaded_results['Std of all Scores']
        worst_f1 = loaded_results['Mean of all scores']['f1Mean'] 
               
    

{'Timeout': 'default', 'Mean of all scores': {'f1Mean': 0.47086637851263075, 'accMean': 0.976235443291746, 'recMean': 0.44868973789856287, 'precMean': 0.6210600313174524}, 'Std of all Scores': {'f1Std': 0.10216837426113554, 'accStd': 0.005724921829971046, 'recStd': 0.1260704332958703, 'precStd': 0.10897177116153353}}
{'Timeout': 0.5, 'Mean of all scores': {'f1Mean': 0.4729493512626144, 'accMean': 0.9763000570866381, 'recMean': 0.45025106795409264, 'precMean': 0.6314211543706032}, 'Std of all Scores': {'f1Std': 0.10415883436271824, 'accStd': 0.005721070229415257, 'recStd': 0.1263919442193845, 'precStd': 0.11022047534371308}}


In [8]:
print("Best Timeout Combination: ", best_timeout)
print("Mean Scores (Best): ", best_mean)
print('Std Scores (Best):', best_std)

Best Timeout Combination:  1
Mean Scores (Best):  {'f1Mean': 0.32251636210055185, 'accMean': 0.9779676457798366, 'recMean': 0.31944281051929146, 'precMean': 0.5290403463187443}
Std Scores (Best): {'f1Std': 0.06989831388841082, 'accStd': 0.014117166990104431, 'recStd': 0.12705967840979823, 'precStd': 0.16405294629630005}


In [9]:
print("worst Timeout Combination: ", worst_timeout)
print("Mean Scores (Worst): ", worst_mean)
print('Std Scores (Worst):', worst_std)

worst Timeout Combination:  1
Mean Scores (Worst):  {'f1Mean': 0.32251636210055185, 'accMean': 0.9779676457798366, 'recMean': 0.31944281051929146, 'precMean': 0.5290403463187443}
Std Scores (Worst): {'f1Std': 0.06989831388841082, 'accStd': 0.014117166990104431, 'recStd': 0.12705967840979823, 'precStd': 0.16405294629630005}


In [31]:

results = {
    'Best score': {
        'Best Timeout': best_timeout,
        'Mean Scores (Best)': best_mean,
        'Std Scores (Best)': best_std,
    },
    
    'Worst score': {
        'Worst Timeout': worst_timeout,
        'Mean Scores (Worst)': worst_mean,
        'Std Scores (Worst)': worst_std,
    },
    
    'Difference': {
        'Accuracy': (best_mean['accMean'] - worst_mean['accMean'])*100,
        'F1 Score': (best_mean['f1Mean'] - worst_mean['f1Mean'])*100,
        'Precision': (best_mean['precMean'] - worst_mean['precMean'])*100,
        'Recall': (best_mean['recMean'] - worst_mean['recMean'])*100
    }
}



with open('../results/ET_unsw_zeek.json', 'w') as f:
    json.dump(results, f, indent=4)